# Basic Neural Networks
This file will go through the same dataset as the basic linear regression file except we will be using a shallow neural network to better fit the data. Again, the majority of this work is either copied or based off of a file that was provided to us by Braxton Osting. All credit due to him for laying this out for us in class.

Import necessary packages

In [2]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import seaborn as sns

This defines the neural network model that we want to use. We will use a size 16 neural network and the ReLU function, which is standard for most neural networks.

In [3]:
model = nn.Sequential(
    nn.Linear(1, 16),
    nn.ReLU(),
    nn.Linear(16, 1),
)
model

Sequential(
  (0): Linear(in_features=1, out_features=16, bias=True)
  (1): ReLU()
  (2): Linear(in_features=16, out_features=1, bias=True)
)

Now we want to count the number of parameters so that we can compare it across models in the future.

In [4]:
sum(p.numel() for p in model.parameters())

49

Now we are going to import the dataset. In this case we need to standardize the predictor or else the neural network will fail to train.

In [5]:
df = sns.load_dataset("mpg").dropna(subset=["horsepower"])
h = df.horsepower.values.astype(np.float32)
y = df.mpg.values.astype(np.float32)

mu, sd = h.mean(), h.std()
X = torch.tensor((h - mu) / sd).unsqueeze(1)   # shape (392, 1)
Y = torch.tensor(y).unsqueeze(1)
print(X.shape, Y.shape)

torch.Size([392, 1]) torch.Size([392, 1])


We will now train our data using Adam, which for now we are instructed to treat as a black box.

In [6]:
opt = torch.optim.Adam(model.parameters(), lr=0.05)
lossfn = nn.MSELoss()

for epoch in range(2000):
    opt.zero_grad()
    loss = lossfn(model(X), Y)
    loss.backward()
    opt.step()
    if epoch % 400 == 0:
        print(f"epoch {epoch:5d}   loss {loss.item():.4f}")

print(f"final MSE {loss.item():.4f}")

epoch     0   loss 613.5772
epoch   400   loss 18.8465
epoch   800   loss 18.4628
epoch  1200   loss 18.4374
epoch  1600   loss 18.4280
final MSE 18.4233


As with the previous example, and all model predictions, we want to know how good it is so we will now calculate RMSE, MAE, and $R^2$ for our model.

In [8]:
with torch.no_grad():
    yhat = model(X).squeeze().numpy()

lin  = np.polyfit(h, y, 1)
quad = np.polyfit(h, y, 2)

def scores(pred):
    r = y - pred
    return (np.sqrt(np.mean(r**2)),
            np.mean(np.abs(r)),
            1 - np.sum(r**2) / np.sum((y - y.mean())**2))

rows = [("linear (L02)",  2, *scores(np.polyval(lin,  h))),
        ("quadratic",     3, *scores(np.polyval(quad, h))),
        (f"NN, m={16}", sum(p.numel() for p in model.parameters()), *scores(yhat))]

insample = pd.DataFrame(rows, columns=["model", "par.", "RMSE", "MAE", "R2"])
print(insample.round({"RMSE": 3, "MAE": 3, "R2": 3}).to_string(index=False))

       model  par.  RMSE   MAE    R2
linear (L02)     2 4.893 3.828 0.606
   quadratic     3 4.357 3.249 0.688
    NN, m=16    49 4.263 3.157 0.701
